# Open Mirroring with Microsoft Fabric using SWAPI – Incremental Changes

This notebook is a **companion** to the `Fabric_OpenMirroring_SWAPI_AllEndpoints` notebook.

Use it **after** you have already run the initial load notebook, so that:

- The Mirrored Database (Open Mirroring / Generic mirror) already exists.
- The landing zone contains initial Parquet batches for:
  - `sw_people`, `sw_planets`, `sw_films`, `sw_species`, `sw_starships`, `sw_vehicles`.
- `_metadata.json` exists in each table folder with a `keyColumns` entry.

This notebook will:

- Fetch fresh data again from SWAPI for each endpoint.
- Create a **small incremental batch** per endpoint that:
  - **Updates** the first entity (rowMarker = 1).
  - **Deletes** the second entity (rowMarker = 2, if present).
- Write those changes as new Parquet files into each table's folder using the
  Open Mirroring required pattern:
  - First column `__rowMarker__`
  - Second column = table-specific key column
  - Rest of the columns = string fields.

Open Mirroring will detect these files and apply the corresponding UPDATE / DELETE
operations to the mirrored tables.


In [ ]:
# STEP 1 – CONFIGURATION
# ----------------------
# Paste the SAME Landing zone URL used in the initial-load notebook.
# Example:
# landing_zone_https = "https://onelake.dfs.fabric.microsoft.com/fabrikam-ws/SWAPI_OpenMirror/Files/LandingZone"

landing_zone_https = "[Landing Zone URL]"  # TODO: replace with your actual URL

def https_to_abfss(url: str) -> str:
    """Convert a OneLake HTTPS URL to an abfss path usable by mssparkutils.fs.

    Example:
      https://onelake.dfs.fabric.microsoft.com/ws-id/item-id/Files/LandingZone
        -> abfss://ws-id@onelake.dfs.fabric.microsoft.com/item-id/Files/LandingZone
    """
    if not url.lower().startswith("https://"):
        raise ValueError("Expected an https://onelake.dfs.fabric.microsoft.com/... URL")

    without_scheme = url[len("https://"):]
    parts = without_scheme.split("/")
    if len(parts) < 3:
        raise ValueError(f"Unexpected Landing zone URL format: {url}")

    host = parts[0]              # onelake.dfs.fabric.microsoft.com
    container = parts[1]         # workspace id / name
    path = "/".join(parts[2:])  # mirrored db id + Files/LandingZone

    return f"abfss://{container}@{host}/{path}"

# Root of the landing zone as abfss URI
landing_zone_root = https_to_abfss(landing_zone_https.rstrip("/"))

# Same endpoint → table mapping used in the initial-load notebook
endpoints_config = {
    "people":   {"table_name": "sw_people",   "key_column": "person_id"},
    "planets":  {"table_name": "sw_planets",  "key_column": "planet_id"},
    "films":    {"table_name": "sw_films",    "key_column": "film_id"},
    "species":  {"table_name": "sw_species",  "key_column": "species_id"},
    "starships":{"table_name": "sw_starships","key_column": "starship_id"},
    "vehicles": {"table_name": "sw_vehicles", "key_column": "vehicle_id"},
}

print("Landing zone (HTTPS):", landing_zone_https)
print("Landing zone (abfss):", landing_zone_root)
print("Configured endpoints:")
for ep, cfg in endpoints_config.items():
    print(f"  - {ep} -> table {cfg['table_name']} (key: {cfg['key_column']})")

In [ ]:
# STEP 2 – DEPENDENCIES & HELPERS
# --------------------------------

# If requests is not installed in this environment, uncomment:
# %pip install requests

import json
import re
import requests

from notebookutils import mssparkutils
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType
)

def fetch_swapi_collection(endpoint: str):
    """Fetch all pages for a given SWAPI endpoint (e.g., 'people', 'planets')."""
    url = f"https://swapi.dev/api/{endpoint}/"
    results = []
    while url:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()
        results.extend(data["results"])
        url = data["next"]
    return results

def extract_id_from_url(url: str) -> int:
    """Extract numeric ID from a SWAPI resource URL, e.g. '.../people/1/'."""
    return int(url.rstrip("/").split("/")[-1])

def normalize_record(rec: dict) -> dict:
    """Convert lists/dicts to JSON strings so we can store everything as strings."""
    out = {}
    for k, v in rec.items():
        if isinstance(v, (list, dict)):
            out[k] = json.dumps(v)
        else:
            out[k] = v
    return out

def get_next_sequence_number(folder: str) -> int:
    """Return the next integer sequence based on existing 20-digit parquet filenames."""
    try:
        files = mssparkutils.fs.ls(folder)
    except Exception:
        # If folder does not exist yet, treat this as first batch
        return 1

    seq_nums = []
    for f in files:
        name = f.name
        if name.endswith(".parquet") and re.match(r"^\d{20}\.parquet$", name):
            seq_nums.append(int(name.split(".")[0]))

    return max(seq_nums) + 1 if seq_nums else 1

def get_sequence_filename(n: int) -> str:
    """Format integer as a 20-digit zero-padded filename."""
    return f"{n:020d}.parquet"

def build_changes_dataframe_for_endpoint(endpoint: str, key_column: str, records: list):
    """Build a DataFrame that simulates an UPDATE and DELETE for the endpoint.

    - UPDATE = first record in the list (rowMarker = 1).
    - DELETE = second record in the list, if present (rowMarker = 2).

    All fields are stored as strings, except:
    - __rowMarker__ (int)
    - key_column (int), derived from the 'url' field.
    """
    if not records:
        raise ValueError(f"No records returned for endpoint '{endpoint}'")

    normalized = [normalize_record(r) for r in records]

    # We will take the first two records for demo purposes
    update_rec = normalized[0]
    delete_rec = normalized[1] if len(normalized) > 1 else None

    # Collect all keys across those two records
    all_keys = set(update_rec.keys())
    if delete_rec is not None:
        all_keys.update(delete_rec.keys())

    if "url" not in all_keys:
        raise ValueError(f"Endpoint '{endpoint}' records do not contain a 'url' field")

    data_columns = sorted(all_keys)  # includes 'url'

    rows = []

    # UPDATE row (rowMarker = 1)
    update_url = update_rec.get("url")
    if update_url:
        update_id = extract_id_from_url(update_url)
        row_update = {
            "__rowMarker__": 1,  # UPDATE
            key_column: update_id,
        }
        for col in data_columns:
            value = update_rec.get(col)
            row_update[col] = None if value is None else str(value)
        rows.append(Row(**row_update))

    # DELETE row (rowMarker = 2), if we have at least 2 records
    if delete_rec is not None:
        delete_url = delete_rec.get("url")
        if delete_url:
            delete_id = extract_id_from_url(delete_url)
            row_delete = {
                "__rowMarker__": 2,  # DELETE
                key_column: delete_id,
            }
            # For DELETE, only key_column is required, but we add other columns as NULL/values
            for col in data_columns:
                value = delete_rec.get(col)
                row_delete[col] = None if value is None else str(value)
            rows.append(Row(**row_delete))

    # Build schema
    schema_fields = [
        StructField("__rowMarker__", IntegerType(), nullable=False),
        StructField(key_column, IntegerType(), nullable=False),
    ]
    for col in data_columns:
        schema_fields.append(StructField(col, StringType(), nullable=True))

    schema = StructType(schema_fields)
    df = spark.createDataFrame(rows, schema=schema)
    return df

print("✅ Helpers loaded. Next: run the 'INCREMENTAL CHANGES – ALL ENDPOINTS' cell.")

In [ ]:
# STEP 3 – INCREMENTAL CHANGES – ALL ENDPOINTS
# --------------------------------------------

for endpoint, cfg in endpoints_config.items():
    table_name = cfg["table_name"]
    key_column = cfg["key_column"]
    table_folder = f"{landing_zone_root}/{table_name}"

    print("\n==============================")
    print(f"Endpoint: {endpoint}")
    print(f"Target table: {table_name} (key: {key_column})")
    print(f"Table folder: {table_folder}")
    print("Fetching current data from SWAPI...")

    records = fetch_swapi_collection(endpoint)
    if not records:
        print("  → No records returned. Skipping endpoint.")
        continue

    print(f"  → {len(records)} records fetched")

    # Build a small change set (1 update + 1 delete)
    df_changes = build_changes_dataframe_for_endpoint(endpoint, key_column, records)
    print("  → Change DataFrame created with", df_changes.count(), "rows")

    # Optionally preview for the first endpoint only
    if endpoint == list(endpoints_config.keys())[0]:
        display(df_changes)

    # Write to a tmp folder, then rename to the next 20-digit sequence
    tmp_path_changes = f"{table_folder}/tmp_changes"

    df_changes.coalesce(1).write.mode("overwrite").parquet(tmp_path_changes)

    files = mssparkutils.fs.ls(tmp_path_changes)
    parquet_files = [f for f in files if f.name.endswith(".parquet")]
    if not parquet_files:
        raise Exception(f"No parquet file written in {tmp_path_changes} for endpoint '{endpoint}'")

    source_file = parquet_files[0].path

    seq_num = get_next_sequence_number(table_folder)
    target_name = get_sequence_filename(seq_num)
    target_path = f"{table_folder}/{target_name}"

    print("  → Moving", source_file, "->", target_path)
    mssparkutils.fs.mv(source_file, target_path)
    mssparkutils.fs.rm(tmp_path_changes, True)

    print("  → Incremental changes parquet created for endpoint:", endpoint)

print("\n✅ Incremental batches written for all endpoints.")
print("Open Mirroring will detect these files and apply UPDATE/DELETE to the mirrored tables.")

## Step 4 – Validate in the Mirrored SQL Endpoint

After the Open Mirroring engine has processed these new Parquet files, you can
        validate changes in the Mirrored Database **SQL analytics endpoint**.

For example, for `sw_people`:

```sql
-- See sample rows
SELECT TOP 10 * FROM dbo.sw_people ORDER BY person_id;

-- Inspect the row with the ID that was updated
SELECT *
FROM dbo.sw_people
WHERE person_id IN (1, 2);  -- adjust IDs as needed
```

Repeat similar queries for the other tables:

- `sw_planets`
- `sw_films`
- `sw_species`
- `sw_starships`
- `sw_vehicles`

You should observe:

- One record updated for each table (first entity in SWAPI for that endpoint).
- One record removed where a DELETE row was generated (second entity).